In [2]:
import os
import sys
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import ndimage
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from abc import ABC, abstractmethod
import cv2
import time
from scipy.optimize import curve_fit

import warnings

# ==============================================================================
# 0. NUMBA ACCELERATION CORE (V2 - BATCH PROCESSING)
# ==============================================================================
from numba import jit, prange

@jit(nopython=True, parallel=True, fastmath=True)
def numba_batch_atr_grid(ch, cv, alphas, betas):
    """
    KERNEL V2: Calcula o score ATR para MÚLTIPLAS configurações de uma vez.
    """
    n_configs = len(alphas)
    rows, cols = ch.shape
    size = ch.size
    
    lh = np.std(ch); lv = np.std(cv)
    if lh < 1e-9: lh = 1e-9
    if lv < 1e-9: lv = 1e-9
    
    scores = np.zeros(n_configs, dtype=np.float64)
    
    for k in prange(n_configs):
        a = alphas[k]; b = betas[k]
        beta_lv = b * lv; alpha_lh = a * lh
        beta_lh = b * lh; alpha_lv = a * lv
        
        count = 0
        for r in range(rows):
            for c in range(cols):
                val_ch = ch[r, c]; val_cv = cv[r, c]
                c1 = (val_cv >= beta_lv) and (val_ch < alpha_lh)
                c2 = (val_ch >= beta_lh) and (val_cv < alpha_lv)
                if c1: count += 1
                if c2: count += 1
        scores[k] = (count / 2.0) / size
    return scores

@jit(nopython=True, fastmath=True)
def numba_stats_moments(arr_flat):
    n = len(arr_flat)
    if n == 0: return 0.0, 0.0, 0.0, 0.0
    mean_val = 0.0
    for i in range(n): mean_val += arr_flat[i]
    mean_val /= n
    m2, m3, m4 = 0.0, 0.0, 0.0
    for i in range(n):
        val = arr_flat[i] - mean_val
        val2 = val * val
        m2 += val2; m3 += val2 * val; m4 += val2 * val2
    var = m2 / n; std_val = np.sqrt(var)
    if m2 == 0: skew_val, kurt_val = 0.0, -3.0
    else:
        skew_val = (m3 / n) / (std_val**3)
        kurt_val = (m4 / n) / (var**2) - 3.0
    return kurt_val, skew_val, std_val, mean_val

@jit(nopython=True, fastmath=True)
def numba_pairwise_stats(mat):
    rows, cols = mat.shape
    if cols < 2: return 0.0, 0.0
    n_pairs = rows * (cols - 1)
    sum_prod = 0.0
    for r in range(rows):
        for c in range(cols - 1): sum_prod += mat[r, c] * mat[r, c+1]
    mean_prod = sum_prod / n_pairs
    sum_sq_diff = 0.0
    for r in range(rows):
        for c in range(cols - 1):
            diff = (mat[r, c] * mat[r, c+1]) - mean_prod
            sum_sq_diff += diff * diff
    return mean_prod, np.sqrt(sum_sq_diff / n_pairs)

@jit(nopython=True, parallel=True, fastmath=True)
def numba_single_atr(ch, cv, alpha, beta):
    lh = np.std(ch); lv = np.std(cv)
    if lh < 1e-9: lh = 1e-9
    if lv < 1e-9: lv = 1e-9
    rows, cols = ch.shape
    beta_lv = beta * lv; alpha_lh = alpha * lh
    beta_lh = beta * lh; alpha_lv = alpha * lv
    cnt = 0
    for i in prange(rows):
        for j in range(cols):
            vch = ch[i, j]; vcv = cv[i, j]
            if (vcv >= beta_lv) and (vch < alpha_lh): cnt += 1
            if (vch >= beta_lh) and (vcv < alpha_lv): cnt += 1
    return cnt / 2.0 / ch.size

# ==============================================================================
# 1. CONFIGURAÇÃO & MATH INTERFACE
# ==============================================================================
DATA_ROOT = "C:/Users/User/NOTEBOOKS-DELL/IMAGE_QUALITY/GEN_IMGS/"
ENABLED_DATASETS = {'LIVE': True, 'CSIQ': True, 'TID2013': True}
ENABLED_METHODS = {'ARQUE': True, 'BRISQUE': True} 

# Grid expandido para arrays Numpy para o Numba
GRID_ALPHAS = [0.05, 0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
GRID_BETAS  = [0.05, 0.1, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

_A_MESH, _B_MESH = np.meshgrid(GRID_ALPHAS, GRID_BETAS)
FLAT_ALPHAS = np.ascontiguousarray(_A_MESH.flatten(), dtype=np.float64)
FLAT_BETAS  = np.ascontiguousarray(_B_MESH.flatten(), dtype=np.float64)

try:
    import cupy as cp
    from cupyx.scipy import ndimage as cp_ndimage
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

class ARQUE_Math_CuPy:
    """
    Motor acelerado por GPU para o ARQUE.
    Mantém os dados na VRAM para cálculo de curvaturas, momentos e matrizes lógicas.
    """
    def calculate_log1p_abs_curvatures(self, img, Dir='hv'):
        # 1. Envia a imagem e kernels para a GPU
        img_gpu = cp.asarray(img, dtype=cp.float32)
        
        if Dir == 'hv':
            kh = cp.array([[1, -2, 1]], dtype=cp.float32)
            kv = cp.array([[1], [-2], [1]], dtype=cp.float32)
        else:
            kh = cp.array([[1, 0, 0], [0, -2, 0], [0, 0, 1]], dtype=cp.float32)
            kv = cp.array([[0, 0, 1], [0, -2, 0], [1, 0, 0]], dtype=cp.float32)
            
        # 2. Convolução e mapeamento logarítmico diretamente no hardware (100% GPU)
        ch = cp.log1p(cp.abs(cp_ndimage.convolve(img_gpu, kh, mode='reflect')))
        cv = cp.log1p(cp.abs(cp_ndimage.convolve(img_gpu, kv, mode='reflect')))
        
        return ch, cv # Retorna ponteiros de memória da GPU

    def get_atr_batch(self, ch, cv):
        # Utiliza os arrays globais FLAT_ALPHAS e FLAT_BETAS definidos no topo do seu script
        n_configs = len(FLAT_ALPHAS)
        size = ch.size
        
        lh = cp.maximum(cp.std(ch), 1e-9)
        lv = cp.maximum(cp.std(cv), 1e-9)
        
        scores = np.zeros(n_configs, dtype=np.float64)
        
        # O loop ocorre na CPU, mas as operações internas são matrizes massivas na GPU
        for k in range(n_configs):
            a, b = FLAT_ALPHAS[k], FLAT_BETAS[k]
            
            beta_lv = b * lv
            alpha_lh = a * lh
            beta_lh = b * lh
            alpha_lv = a * lv
            
            # Operação bitwise vetorizada (resolve a matriz inteira em um ciclo de clock)
            c1 = (cv >= beta_lv) & (ch < alpha_lh)
            c2 = (ch >= beta_lh) & (cv < alpha_lv)
            
            # Soma os booleanos e calcula o score
            cnt = cp.sum(c1) + cp.sum(c2)
            scores[k] = float(cnt) / (2.0 * size)
            
        return scores # Retorna array NumPy (CPU) pronto para o Spearman no treino

    def get_atr_single(self, ch, cv, alpha, beta):
        size = ch.size
        lh = cp.maximum(cp.std(ch), 1e-9)
        lv = cp.maximum(cp.std(cv), 1e-9)
        
        beta_lv = beta * lv
        alpha_lh = alpha * lh
        beta_lh = beta * lh
        alpha_lv = alpha * lv
        
        c1 = (cv >= beta_lv) & (ch < alpha_lh)
        c2 = (ch >= beta_lh) & (cv < alpha_lv)
        
        cnt = cp.sum(c1) + cp.sum(c2)
        
        # .get() converte o float da GPU de volta para a CPU
        return float((cnt / (2.0 * size)).get())

    def get_nss_features(self, ch, cv):
        f = []
        for m in [ch, cv]:
            mean_val = cp.mean(m)
            std_val = cp.std(m)
            
            if std_val == 0:
                skew_val, kurt_val = 0.0, -3.0
            else:
                m_centered = m - mean_val
                var = std_val ** 2
                skew_val = cp.mean(m_centered ** 3) / (std_val ** 3)
                kurt_val = cp.mean(m_centered ** 4) / (var ** 2) - 3.0
                
            f.extend([float(kurt_val.get()), float(skew_val.get()), float(std_val.get()), float(mean_val.get())])
            
        for m in [ch, cv]:
            if m.shape[1] < 2:
                mean_p, std_p = 0.0, 0.0
            else:
                prod = m[:, :-1] * m[:, 1:]
                mean_p = cp.mean(prod)
                std_p = cp.std(prod)
                
            f.extend([float(mean_p.get()), float(std_p.get())])
            
        return f
        
class ARQUE_Math_Numba_V2:
    def calculate_log1p_abs_curvatures(self, img, Dir='hv'):
        if Dir =='hv': kh=np.array([[1,-2,1]]); kv=np.array([[1],[-2],[1]])
        else: kh=np.array([[1,0,0],[0,-2,0],[0,0,1]]); kv=np.array([[0,0,1],[0,-2,0],[1,0,0]])
        img = img.astype(np.float32)
        ch = np.log1p(np.abs(ndimage.convolve(img, kh, mode='reflect')))
        cv = np.log1p(np.abs(ndimage.convolve(img, kv, mode='reflect')))
        return ch, cv

    def get_atr_batch(self, ch, cv):
        return numba_batch_atr_grid(ch, cv, FLAT_ALPHAS, FLAT_BETAS)

    def get_atr_single(self, ch, cv, alpha, beta):
        return numba_single_atr(ch, cv, float(alpha), float(beta))

    def get_nss_features(self, ch, cv):
        f = []
        for m in [ch, cv]:
            k, s, std, mean = numba_stats_moments(m.flatten())
            f.extend([k, s, std, mean])
        for m in [ch, cv]:
            mean_p, std_p = numba_pairwise_stats(m)
            f.extend([mean_p, std_p])
        return f

USE_GPU = True # Troque por um argumento do argparse no futuro

if USE_GPU and CUPY_AVAILABLE:
    print(">>> Iniciando motor ARQUE em GPU (CuPy)...")
    math_core = ARQUE_Math_CuPy()
else:
    print(">>> Iniciando motor ARQUE em CPU (Numba)...")
    math_core = ARQUE_Math_Numba_V2()

# ==============================================================================
# 3. ENGINE DE I/O (BASE)
# ==============================================================================
def load_live(root):
    print(f"   [I/O] Loading LIVE from {root}...")
    try: dmos = sio.loadmat(os.path.join(root, 'dmos.mat'))['dmos'].flatten()
    except: return []
    offs = {'jp2k':0, 'jpeg':227, 'wn':460, 'gblur':634, 'fastfading':808}
    lens = {'jp2k':227, 'jpeg':233, 'wn':174, 'gblur':174, 'fastfading':174}
    data = []
    for t in lens.keys():
        base = os.path.join(root, t)
        for i in range(lens[t]):
            fname = f"img{i+1}.bmp"
            if not os.path.exists(os.path.join(base, fname)): fname = f"img{i+1}.jpg"
            fpath = os.path.join(base, fname)
            if os.path.exists(fpath): data.append({'path': fpath, 'type': t, 'score': dmos[offs[t] + i]})
    return data

def load_csiq(root):
    print(f"   [I/O] Loading CSIQ from {root}...")
    csv_path = os.path.join(root, 'csiq_scores_by_image.csv')
    if not os.path.exists(csv_path): return []
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    data = []; img_root = os.path.join(root, 'dst_imgs')
    target_map = {'noise':'wn', 'awgn':'wn', 'blur':'gblur', 'jpeg':'jpeg', 'jpeg2000':'jp2k'}
    folder_map = {'noise': 'awgn', 'blur': 'blur', 'jpeg': 'jpeg', 'jpeg2000': 'jpeg2000'}
    for _, row in df.iterrows():
        dtype = str(row.get('dst_type', '')).strip().lower()
        if dtype in target_map:
            my_type = target_map[dtype]
            img_b = str(row['image']).strip(); lev = str(row['dst_lev']).strip()
            ftag = dtype if 'noise' not in dtype and 'awgn' not in dtype else 'AWGN'
            if '2000' in dtype: ftag = 'jpeg2000'
            fname = f"{img_b}.{ftag}.{lev}.png"
            fdir = os.path.join(img_root, folder_map.get(dtype, dtype))
            found = None
            if os.path.exists(os.path.join(fdir, fname)): found = os.path.join(fdir, fname)
            elif os.path.exists(os.path.join(fdir, fname.lower())): found = os.path.join(fdir, fname.lower())
            if found: data.append({'path': found, 'type': my_type, 'score': float(row['dmos'])})
    return data

def load_tid2013(root, max_per_type=100, seed=42):
    print(f"   [I/O] Loading TID2013 (Subset: max {max_per_type}/type) from {root}...")
    mos_file = os.path.join(root, 'mos_with_names.txt')
    if not os.path.exists(mos_file): 
        print("      [Error] mos_with_names.txt not found.")
        return []
    tid_map = {1:'wn', 8:'gblur', 10:'jpeg', 11:'jp2k'} 
    img_folder = os.path.join(root, 'distorted_images')
    grouped_data = {}
    with open(mos_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2: continue
            score = float(parts[0])
            fname = parts[1]
            try:
                type_id = int(fname.split('_')[1])
                t_str = tid_map.get(type_id, f"type_{type_id:02d}")
                full_path = os.path.join(img_folder, fname)
                if os.path.exists(full_path):
                    item = {'path': full_path, 'type': t_str, 'score': score}
                    if type_id not in grouped_data: grouped_data[type_id] = []
                    grouped_data[type_id].append(item)
            except: pass
    final_data = []
    rng = np.random.RandomState(seed)
    print(f"      -> Sampling {max_per_type} images from each of the {len(grouped_data)} distortion types...")
    for tid, items in grouped_data.items():
        if len(items) > max_per_type:
            selected = rng.choice(items, max_per_type, replace=False)
            final_data.extend(selected)
        else:
            final_data.extend(items)
    return final_data

LOADERS = {'LIVE': load_live, 'CSIQ': load_csiq, 'TID2013': load_tid2013}

# ==============================================================================
# 4. IMPLEMENTAÇÃO DOS MÉTODOS
# ==============================================================================
class GenericIQA(ABC):
    @abstractmethod
    def train(self, train_data): pass
    @abstractmethod
    def predict(self, test_data): pass

class ARQUE_Integrated(GenericIQA):
    def __init__(self):
        self.name = "ARQUE"
        self.optimized_params = {}
        self.clf = None
        self.svrs = {}
        self.le = LabelEncoder()

    def train(self, train_data):
        # 1. AUTO-CALIBRAÇÃO (VETORIZADA)
        tic = time.time()
        unique_types = np.unique([d['type'] for d in train_data])
        print(f"        [ARQUE] 1. Calculating optimal ATR parameters (Fast Batch Grid)...")
        
        for t in unique_types:
            subset = [d for d in train_data if d['type'] == t]
            if len(subset) < 5: 
                self.optimized_params[t] = {'alpha': 1.0, 'beta': 1.0}; continue
            
            true_scores, all_atr_results = [], []
            for item in subset:
                img = cv2.imread(item['path'], cv2.IMREAD_GRAYSCALE)
                if img is None: continue
                ch, cv = math_core.calculate_log1p_abs_curvatures(img)
                atr_vector = math_core.get_atr_batch(ch, cv)
                all_atr_results.append(atr_vector)
                true_scores.append(item['score'])
            
            if not all_atr_results: continue
            X_grid = np.array(all_atr_results)
            y_true = np.array(true_scores)
            
            best_rho = -1; best_idx = 0
            n_configs = X_grid.shape[1]
            for col_idx in range(n_configs):
                preds = X_grid[:, col_idx]
                if np.std(preds) > 1e-9:
                    r = abs(spearmanr(y_true, preds)[0])
                    if not np.isnan(r) and r > best_rho:
                        best_rho = r; best_idx = col_idx
            self.optimized_params[t] = {'alpha': FLAT_ALPHAS[best_idx], 'beta': FLAT_BETAS[best_idx]}

        # 2. TREINAMENTO
        print("        [ARQUE] 2. Training Specialists...")
        X_clf, y_clf_labels = [], []
        X_svr_data = {t: {'X': [], 'y': []} for t in unique_types}
        sorted_keys = sorted(self.optimized_params.keys())
        
        for item in train_data:
            t = item['type']
            img = cv2.imread(item['path'], cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            ch, cv = math_core.calculate_log1p_abs_curvatures(img)
            p = self.optimized_params[t]
            atr = math_core.get_atr_single(ch, cv, p['alpha'], p['beta'])
            nss = math_core.get_nss_features(ch, cv)
            
            feat_vec = [atr] + nss 
            X_svr_data[t]['X'].append(feat_vec)
            X_svr_data[t]['y'].append(item['score'])
            
            atrs_all = []
            for k in sorted_keys:
                pk = self.optimized_params[k]
                atrs_all.append(math_core.get_atr_single(ch, cv, pk['alpha'], pk['beta']))
            X_clf.append(atrs_all + nss)
            y_clf_labels.append(t)

        self.le.fit(unique_types)
        y_enc = self.le.transform(y_clf_labels)
        self.clf = RandomForestClassifier(n_estimators=100, random_state=42)
        self.clf.fit(X_clf, y_enc)
        
        for t, data in X_svr_data.items():
            if len(data['X']) >= 5:
                regr = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=100, gamma='scale'))
                regr.fit(data['X'], data['y'])
                self.svrs[t] = regr
        
    def predict(self, test_data):
        print(f"        [ARQUE] Predicting {len(test_data)} samples...")
        y_pred, y_true = [], []
        sorted_keys = sorted(self.optimized_params.keys())
        for item in test_data:
            img = cv2.imread(item['path'], cv2.IMREAD_GRAYSCALE)
            if img is None: y_pred.append(50.0); y_true.append(item['score']); continue
            ch, cv = math_core.calculate_log1p_abs_curvatures(img)
            atrs_all = []
            for k in sorted_keys:
                pk = self.optimized_params[k]
                atrs_all.append(math_core.get_atr_single(ch, cv, pk['alpha'], pk['beta']))
            nss = math_core.get_nss_features(ch, cv)
            
            probs = self.clf.predict_proba([atrs_all + nss])[0]
            val = 0.0; w_sum = 0.0
            for idx, prob in enumerate(probs):
                if prob < 0.01: continue
                c_name = self.le.inverse_transform([idx])[0]
                if c_name in self.svrs:
                    pk = self.optimized_params[c_name]
                    atr_spec = math_core.get_atr_single(ch, cv, pk['alpha'], pk['beta'])
                    feat = np.array([[atr_spec] + nss])
                    val += prob * self.svrs[c_name].predict(feat)[0]
                    w_sum += prob
            y_pred.append(val / w_sum if w_sum > 0 else 50.0)
            y_true.append(item['score'])
        return np.array(y_pred), np.array(y_true)

class Baseline_NSS(GenericIQA):
    """
    BASELINE (NSS + SVR):
    Funciona como proxy para o BRISQUE. 
    Usa apenas estatísticas de cena natural (NSS) sem a geometria ATR.
    """
    def __init__(self): 
        self.name = "Baseline_NSS"
        self.svr = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=100, gamma='scale'))
            
    def train(self, train_data): 
        print(f"        [Baseline] Training SVR on NSS features ({len(train_data)} samples)...")
        X, y = [], []
        for item in train_data:
            img = cv2.imread(item['path'], cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            ch, cv = math_core.calculate_log1p_abs_curvatures(img)
            nss = math_core.get_nss_features(ch, cv)
            X.append(nss)
            y.append(item['score'])
        if len(X) > 0: self.svr.fit(X, y)
    
    def predict(self, test_data):
        print(f"        [Baseline] Predicting {len(test_data)} samples...")
        X, y_true = [], []
        for item in test_data:
            img = cv2.imread(item['path'], cv2.IMREAD_GRAYSCALE)
            if img is None: 
                if len(X) > 0: X.append(X[-1])
                else: X.append([0]*12)
                y_true.append(item['score']); continue
            ch, cv = math_core.calculate_log1p_abs_curvatures(img)
            nss = math_core.get_nss_features(ch, cv)
            X.append(nss)
            y_true.append(item['score'])
        if len(X) == 0: return np.array([]), np.array([])
        return self.svr.predict(X), np.array(y_true)

METHODS_MAP = {
    'ARQUE': ARQUE_Integrated, 
    'BRISQUE': Baseline_NSS
}

# Ignorar warnings de runtime (overflow) durante o ajuste, pois tratamos isso
warnings.filterwarnings("ignore", category=RuntimeWarning) 

def evaluate_with_logistic_mapping(y_pred, y_true, method_name="Method"):
    """
    VERSÃO ENGENHARIA: Mapeamento Linear Simples (y = mx + c).
    """
    y_p = np.array(y_pred, dtype=float).flatten()
    y_t = np.array(y_true, dtype=float).flatten()

    # Ajuste Linear (Polinômio de grau 1)
    m, c = np.polyfit(y_p, y_t, 1)
    y_pred_mapped = m * y_p + c

    plcc, _ = pearsonr(y_pred_mapped, y_t)
    srocc, _ = spearmanr(y_pred, y_t) 
    rmse = np.sqrt(mean_squared_error(y_t, y_pred_mapped))
    
    return plcc, srocc, rmse, y_pred_mapped
    
# ==============================================================================
# 5. EXECUÇÃO COMPLETA (DETAILED REPORT PER FOLD X DISTORTION)
# ==============================================================================
if __name__ == "__main__":
    print("=== IQA BENCHMARK: FINALE (RAW METRICS PER FOLD x DISTORTION) ===")
    
    N_FOLDS = 5
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    for d_name, enabled in ENABLED_DATASETS.items():
        if not enabled: continue
        print(f"\n{'#'*60}")
        print(f">>> DATASET: {d_name}")
        print(f"{'#'*60}")
        
        loader_func = LOADERS.get(d_name)
        dataset = loader_func(os.path.join(DATA_ROOT, d_name))
        
        if not dataset: 
            print(f"   [Warning] Dataset {d_name} is empty or not found.")
            continue
        
        X_dummy = np.zeros(len(dataset))
        y_types = np.array([d['type'] for d in dataset])
        
        for m_name, m_enabled in ENABLED_METHODS.items():
            if not m_enabled: continue
            print(f"\n--- Method: {m_name} [RAW METRICS MODE] ---")
            
            # Armazenar métricas para agregação final (Opcional, mas útil para ver resumo)
            dist_results_agg = {}
            
            for fold_i, (train_idx, test_idx) in enumerate(skf.split(X_dummy, y_types)):
                train_set = [dataset[i] for i in train_idx]
                test_set  = [dataset[i] for i in test_idx]
                
                model = METHODS_MAP[m_name]()
                model.train(train_set)
                y_pred_fold, y_true_fold = model.predict(test_set)

                # =======================================================
                # MODO RAW (SEM MAPEAMENTO)
                # =======================================================
                y_pred_final = y_pred_fold 
                
                # Análise POR TIPO DE DISTORÇÃO dentro deste Fold
                test_types = y_types[test_idx]
                unique_distortions = np.unique(test_types)
                
                print(f"\n   >>> FOLD {fold_i+1} DETAILED RESULTS <<<")
                print(f"   {'Distortion':<20} | {'PLCC':<10} | {'SROCC':<10} | {'RMSE':<10}")
                print(f"   {'-'*60}")

                for dtype in unique_distortions:
                    # Filtra índices deste tipo
                    mask = (test_types == dtype)
                    sub_pred = y_pred_final[mask]
                    sub_true = y_true_fold[mask]
                    
                    if len(sub_pred) < 2: 
                        # Evita erro se houver só 1 imagem no split
                        print(f"   {dtype:<20} | {'N/A':<10} | {'N/A':<10} | {'N/A':<10}")
                        continue
                        
                    plcc, _ = pearsonr(sub_pred, sub_true)
                    srocc, _ = spearmanr(sub_pred, sub_true)
                    rmse = np.sqrt(mean_squared_error(sub_true, sub_pred))
                    
                    # IMPRESSÃO IMEDIATA DENTRO DO FOLD
                    print(f"   {dtype:<20} | {plcc:.4f}     | {srocc:.4f}     | {rmse:.4f}")
                    
                    # Guardar para tabela final global
                    if dtype not in dist_results_agg: dist_results_agg[dtype] = []
                    dist_results_agg[dtype].append({'plcc': plcc, 'srocc': srocc, 'rmse': rmse})

            # =======================================================
            # RELATÓRIO GLOBAL FINAL (MÉDIA E CV)
            # =======================================================
            print(f"\n   >>> AGGREGATED STATISTICS (Mean & CV over {N_FOLDS} Folds) <<<")
            report_data = []
            for dtype, metrics_list in dist_results_agg.items():
                df_temp = pd.DataFrame(metrics_list)
                plcc_m = df_temp['plcc'].mean()
                srocc_m = df_temp['srocc'].mean()
                rmse_m = df_temp['rmse'].mean()
                row = {
                    'Distortion': dtype,
                    'PLCC_Mean': plcc_m,
                    'PLCC_CV': df_temp['plcc'].std() / abs(plcc_m) if plcc_m != 0 else np.nan,
                    'SROCC_Mean': srocc_m,
                    'SROCC_CV': df_temp['srocc'].std() / abs(srocc_m) if srocc_m != 0 else np.nan,
                    'RMSE_Mean': rmse_m,
                    'RMSE_CV': df_temp['rmse'].std() / abs(rmse_m) if rmse_m != 0 else np.nan
                }
                report_data.append(row)
            
            df_report = pd.DataFrame(report_data)
            if not df_report.empty:
                pd.set_option('display.float_format', '{:.4f}'.format)
                df_report = df_report.set_index('Distortion').sort_index()
                print(df_report[['PLCC_Mean', 'PLCC_CV', 'SROCC_Mean', 'SROCC_CV', 'RMSE_Mean', 'RMSE_CV']])
            else:
                print("   [Warning] No valid metrics computed.")
            
            print(f"   {'='*80}")


>>> Iniciando motor ARQUE em CPU (Numba)...
=== IQA BENCHMARK: FINALE (RAW METRICS PER FOLD x DISTORTION) ===

############################################################
>>> DATASET: LIVE
############################################################
   [I/O] Loading LIVE from C:/Users/User/NOTEBOOKS-DELL/IMAGE_QUALITY/GEN_IMGS/LIVE...

--- Method: ARQUE [RAW METRICS MODE] ---
        [ARQUE] 1. Calculating optimal ATR parameters (Fast Batch Grid)...
        [ARQUE] 2. Training Specialists...
        [ARQUE] Predicting 178 samples...

   >>> FOLD 1 DETAILED RESULTS <<<
   Distortion           | PLCC       | SROCC      | RMSE      
   ------------------------------------------------------------
   fastfading           | 0.8566     | 0.8364     | 10.0183
   gblur                | 0.8790     | 0.8857     | 8.1628
   jp2k                 | 0.9261     | 0.9168     | 8.0471
   jpeg                 | 0.9228     | 0.9300     | 7.8092
   wn                   | 0.9626     | 0.9694     | 6.1008
 